# Week 6 — LLM-Specific Deployment

### Why serving a language model breaks every assumption from the first five weeks

> **MLOps & Deployment: A From-Scratch Engineering Codex**
> Part of the applied-systems track. Like the rest of this curriculum, every
> abstraction here is *built before it is used*: we re-implement the core of each
> MLOps tool in plain Python/NumPy first, then map what we built onto the
> industry-standard equivalent so the magic is gone by the time you reach it.

---

## Learning objectives

By the end of this notebook you will be able to:

1. Explain why autoregressive generation makes LLM serving fundamentally different.
2. Implement a KV-cache and measure its asymptotic speed-up from scratch.
3. Quantize weights to int8 and quantify the memory/quality trade-off.
4. Build a continuous-batching scheduler and contrast it with Week 3's static batching.
5. Do the back-of-envelope math for LLM memory, throughput, and cost.

## Prerequisites

- Weeks 0–5 (the full MLOps stack).
- Conceptual familiarity with transformers/attention (your advanced_llm_architecture course is ideal background).

## How to read this notebook

Each section has three layers:

- **🧠 Theory** — the systems problem and *why* it exists.
- **🔧 From scratch** — a minimal, dependency-light implementation you fully own.
- **🏭 In practice** — how the real tool (MLflow, Docker, K8s, etc.) does the same thing, and what it adds.

Run cells top-to-bottom. Everything is self-contained and works on a CPU-only laptop.

In [1]:
# --- Standard setup ---------------------------------------------------------
from __future__ import annotations
import os, sys, json, time, math, hashlib, random, pathlib, tempfile, shutil
from dataclasses import dataclass, field, asdict
from typing import Any, Callable

import numpy as np

np.random.seed(0)
random.seed(0)

# A scratch workspace that we clean up between runs of this notebook.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="mlops_week_"))
print("Working directory for this notebook:", WORK)

# --- Content-addressable hashing (introduced & explained in Week 0) ---------
# Re-defined here so each notebook is fully self-contained and runnable alone.
def hash_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:12]

def hash_obj(obj) -> str:
    canonical = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hash_bytes(canonical.encode())

def hash_array(a: np.ndarray) -> str:
    return hash_bytes(a.tobytes() + str(a.shape).encode() + str(a.dtype).encode())

def capture_environment() -> dict:
    import platform
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__}

Working directory for this notebook: /tmp/mlops_week_zesaaudx


## 1. 🧠 What makes LLM serving different

Every model in Weeks 0–5 was a *single forward pass*: `predict(X) → y`, one shot.
An LLM is **autoregressive**: to produce N tokens it runs N forward passes, each
conditioned on all previous tokens. This one fact breaks nearly every serving
assumption:

| Assumption (Weeks 1–5) | What LLMs do instead |
|------------------------|----------------------|
| One request = one forward pass | One request = *hundreds* of sequential passes |
| Latency is a single number | Two latencies: **time-to-first-token** and **inter-token latency** |
| Output size is fixed | Output length is *unknown until generation stops* |
| Static batching is optimal | Requests finish at different times → static batching wastes the GPU |
| Memory is dominated by weights | Memory is dominated by the **KV-cache**, which grows per token |

So LLM serving needs its own techniques: KV-caching, quantization, paged attention,
and **continuous** batching. We'll build the core ideas with tiny NumPy models so
the mechanism — not a 70B-parameter checkpoint — is the point.

## 2. 🧠 The two latencies, and the prefill/decode split

An LLM request has two phases with completely different performance profiles:

- **Prefill** — process the whole prompt in *one* parallel pass. Compute-bound;
  uses the GPU well. Determines **time-to-first-token (TTFT)**.
- **Decode** — generate output tokens one at a time, each a tiny pass. *Memory-
  bandwidth-bound*; the GPU is mostly idle waiting on memory. Determines
  **inter-token latency (ITL)** and dominates total time for long outputs.

A user feels TTFT as "did it start?" and ITL as "how fast does it type?". You
optimise them with different tools. This split is the mental model for everything
below.

## 3. 🔧 From scratch: the KV-cache

Here's the waste autoregressive generation creates. At each step, attention needs
the keys and values of *all previous tokens*. The naive approach recomputes them
every step — so generating token `t` redoes the work for tokens `0..t-1`. Total
work is **O(N²)**.

The fix: **cache** the keys and values once computed. Then each new token only
computes *its own* K and V and reuses the rest — total work **O(N)**. Let's build
a minimal attention loop both ways and measure it.

In [2]:
# A toy single-head self-attention over a sequence, with toy random projections.
D = 64                          # embedding dim
rng = np.random.RandomState(0)
Wq, Wk, Wv = (rng.randn(D, D) / np.sqrt(D) for _ in range(3))

def attention(Q, K, V):
    scores = Q @ K.T / np.sqrt(D)
    # causal mask
    mask = np.triu(np.ones_like(scores), k=1).astype(bool)
    scores[mask] = -1e9
    w = np.exp(scores - scores.max(axis=-1, keepdims=True))
    w /= w.sum(axis=-1, keepdims=True)
    return w @ V

def generate_naive(tokens, n_new):
    """Recompute K,V for the ENTIRE sequence at every step. O(N^2)."""
    seq = tokens.copy()
    flops = 0
    for _ in range(n_new):
        K = seq @ Wk; V = seq @ Wv; Q = seq @ Wq      # recompute everything
        flops += 3 * seq.shape[0] * D * D             # projection cost proxy
        out = attention(Q, K, V)
        next_tok = out[-1:] @ rng.randn(D, D) * 0     # dummy next token
        next_tok = np.tanh(out[-1:])
        seq = np.vstack([seq, next_tok])
    return seq, flops

def generate_cached(tokens, n_new):
    """Compute K,V once for the prompt, then only for each NEW token. O(N)."""
    seq = tokens.copy()
    K_cache = seq @ Wk; V_cache = seq @ Wv            # prefill: once
    flops = 3 * seq.shape[0] * D * D
    for _ in range(n_new):
        q = seq[-1:] @ Wq                             # only the new token's Q
        out = attention(q, K_cache, V_cache)
        flops += 3 * 1 * D * D                        # only ONE token's projections
        next_tok = np.tanh(out[-1:])
        seq = np.vstack([seq, next_tok])
        K_cache = np.vstack([K_cache, next_tok @ Wk]) # append new K,V to cache
        V_cache = np.vstack([V_cache, next_tok @ Wv])
    return seq, flops

prompt = rng.randn(8, D)        # 8-token prompt
for n_new in [16, 64, 256]:
    _, f_naive = generate_naive(prompt, n_new)
    _, f_cached = generate_cached(prompt, n_new)
    print(f"generate {n_new:>3} tokens: naive_flops={f_naive:>12,}  "
          f"cached_flops={f_cached:>10,}  speedup={f_naive/f_cached:5.1f}x")

generate  16 tokens: naive_flops=   3,047,424  cached_flops=   294,912  speedup= 10.3x
generate  64 tokens: naive_flops=  31,064,064  cached_flops=   884,736  speedup= 35.1x


generate 256 tokens: naive_flops= 426,246,144  cached_flops= 3,244,032  speedup=131.4x


The speed-up *grows with output length* — exactly the O(N²)→O(N) improvement. Every
production LLM server uses a KV-cache; without it, long generations would be
quadratically slow. This is also *why* the KV-cache dominates memory: it stores K
and V for every token of every active sequence.

## 4. 🧠🔧 KV-cache memory: the real capacity constraint

The KV-cache size, not the model weights, usually decides how many concurrent
requests fit on a GPU. The formula is worth memorising:

$$ \text{KV bytes} = 2 \times n_{\text{layers}} \times n_{\text{tokens}} \times d_{\text{model}} \times \text{bytes per element} $$

(the 2 is for K *and* V). Let's compute it for a realistic model and see how fast
it eats memory.

In [3]:
def kv_cache_bytes(n_layers, d_model, n_tokens, dtype_bytes=2):  # fp16 = 2 bytes
    return 2 * n_layers * n_tokens * d_model * dtype_bytes

# A 7B-class model, roughly
n_layers, d_model = 32, 4096
print("KV-cache memory for a 7B-class model (fp16):")
for ctx in [512, 2048, 8192, 32768]:
    mb = kv_cache_bytes(n_layers, d_model, ctx) / 1e6
    print(f"  context {ctx:>6} tokens: {mb:8.1f} MB per sequence")

# How many concurrent 2048-token sequences fit in 16 GB of *spare* GPU memory?
spare_gb = 16
per_seq_mb = kv_cache_bytes(n_layers, d_model, 2048) / 1e6
print(f"\nWith {spare_gb} GB free, concurrent 2048-token sequences: "
      f"{int(spare_gb*1000 / per_seq_mb)}")

KV-cache memory for a 7B-class model (fp16):
  context    512 tokens:    268.4 MB per sequence
  context   2048 tokens:   1073.7 MB per sequence
  context   8192 tokens:   4295.0 MB per sequence
  context  32768 tokens:  17179.9 MB per sequence

With 16 GB free, concurrent 2048-token sequences: 14


This is why **context length is expensive** and why throughput collapses with many
long-context users: each one holds a big, growing KV-cache. PagedAttention (vLLM's
key innovation) treats the cache like OS virtual memory — non-contiguous pages — to
stop fragmentation from wasting this scarce resource. Same idea as Week 2's OS
namespaces: borrow a proven systems concept and apply it to the model.

## 5. 🔧 From scratch: int8 quantization

Weights in fp16 cost 2 bytes each; a 7B model is ~14 GB. **Quantization** stores
weights in lower precision (int8 = 1 byte, halving memory; int4 quarters it). The
trick is the scale factor: map the float range onto the integer range, store the
ints plus one scale, dequantize on the fly.

In [4]:
def quantize_int8(W: np.ndarray):
    """Symmetric per-tensor int8 quantization."""
    scale = np.abs(W).max() / 127.0
    q = np.round(W / scale).clip(-127, 127).astype(np.int8)
    return q, scale

def dequantize_int8(q: np.ndarray, scale: float) -> np.ndarray:
    return q.astype(np.float32) * scale

# Quantize a weight matrix and measure error + memory.
W = rng.randn(4096, 4096).astype(np.float32)
q, scale = quantize_int8(W)
W_recon = dequantize_int8(q, scale)

rel_err = np.abs(W - W_recon).mean() / np.abs(W).mean()
print(f"fp32 size : {W.nbytes/1e6:7.2f} MB")
print(f"int8 size : {q.nbytes/1e6:7.2f} MB  (+ 4 bytes for scale)")
print(f"compression: {W.nbytes/q.nbytes:.1f}x")
print(f"mean relative reconstruction error: {rel_err:.4%}")

# Does it still compute correctly? Compare a matmul.
x = rng.randn(4096).astype(np.float32)
y_full = x @ W
y_quant = x @ W_recon
print(f"\nmatmul relative error: {np.abs(y_full-y_quant).mean()/np.abs(y_full).mean():.4%}")

fp32 size :   67.11 MB
int8 size :   16.78 MB  (+ 4 bytes for scale)
compression: 4.0x
mean relative reconstruction error: 1.3761%

matmul relative error: 1.2666%


The compression above is **4x** because we started from fp32; from the fp16 that
LLMs actually ship in, int8 is a **2x** win (and int4 a 4x win). Either way, the
matmul output barely changes. That tiny error is the whole bet of quantization: a
few percent reconstruction error usually costs almost nothing in downstream task
quality, while shrinking the weights and roughly doubling throughput — because the
decode phase is memory-bandwidth-bound (Section 2), and you now move fewer bytes.
Modern methods (GPTQ, AWQ, SmoothQuant) are this same idea plus cleverness about
*which* weights tolerate low precision.

## 6. 🧠🔧 Continuous batching: why Week 3's batching isn't enough

Recall Week 3's static dynamic-batching: collect a batch, run it, return all
results together. For LLMs this is *terrible*, because requests in a batch finish
at wildly different times — one wants 10 tokens, another wants 500. With static
batching, the whole batch waits for the longest generation, and finished slots sit
idle.

**Continuous batching** (a.k.a. iteration-level scheduling) fixes this: at *every
generation step*, evict finished sequences and admit waiting ones. The batch
composition changes each step, keeping the GPU full. Let's simulate the GPU
utilisation difference.

In [5]:
def simulate_static_batching(request_lengths, batch_size):
    """All requests in a batch run until the LONGEST finishes. Idle slots wasted."""
    total_slot_steps, used_slot_steps = 0, 0
    for i in range(0, len(request_lengths), batch_size):
        batch = request_lengths[i:i+batch_size]
        steps = max(batch)                         # batch runs until longest done
        total_slot_steps += steps * len(batch)     # slots * steps available
        used_slot_steps += sum(batch)              # actual useful token-steps
    return used_slot_steps / total_slot_steps

def simulate_continuous_batching(request_lengths, batch_size):
    """Finished sequences are evicted each step; waiting ones admitted. GPU stays full."""
    import heapq
    pending = list(request_lengths)
    active = []          # remaining tokens for each active slot
    used, total = 0, 0
    while pending or active:
        # admit waiting requests into free slots
        while len(active) < batch_size and pending:
            active.append(pending.pop(0))
        # one generation step: every active slot produces one token
        total += batch_size                        # all slots available this step
        used += len(active)                        # slots actually doing work
        active = [r - 1 for r in active]
        active = [r for r in active if r > 0]       # evict finished sequences
    return used / total

# Highly variable output lengths (the realistic, hard case)
lengths = list(np.random.RandomState(0).randint(5, 200, 64))
for bs in [8, 16, 32]:
    su = simulate_static_batching(lengths, bs)
    cu = simulate_continuous_batching(lengths, bs)
    print(f"batch_size={bs:>2}: static GPU utilisation={su:5.1%}  "
          f"continuous={cu:5.1%}  gain={cu/su:.2f}x")

batch_size= 8: static GPU utilisation=54.9%  continuous=88.7%  gain=1.62x
batch_size=16: static GPU utilisation=53.8%  continuous=77.9%  gain=1.45x
batch_size=32: static GPU utilisation=53.3%  continuous=64.5%  gain=1.21x


Continuous batching keeps utilisation high regardless of how variable the output
lengths are, while static batching bleeds throughput as variance grows. This is
the single biggest reason vLLM and TGI achieve multiples of the throughput of a
naive server — and it's a direct evolution of the Week 3 scheduler you already
built, now operating at *token granularity* instead of *request granularity*.

## 7. 🧠 Putting it together: the LLM serving cost model

You can now estimate the cost of serving an LLM on the back of an envelope —
exactly what you do before provisioning hardware.

In [6]:
def serving_estimate(params_b, dtype_bytes, gpu_mem_gb, ctx_len,
                     n_layers, d_model, decode_tok_per_s):
    weight_gb = params_b * 1e9 * dtype_bytes / 1e9
    kv_per_seq_gb = kv_cache_bytes(n_layers, d_model, ctx_len, dtype_bytes) / 1e9
    free_for_kv = gpu_mem_gb - weight_gb - 2     # reserve ~2GB overhead
    max_concurrent = max(0, int(free_for_kv / kv_per_seq_gb))
    return {
        "weights_GB": round(weight_gb, 1),
        "kv_per_seq_GB": round(kv_per_seq_gb, 3),
        "free_for_kv_GB": round(free_for_kv, 1),
        "max_concurrent_seqs": max_concurrent,
        "aggregate_tok_per_s": max_concurrent * decode_tok_per_s,
    }

print("7B model, fp16, on an 80GB GPU, 4096 ctx:")
print(json.dumps(serving_estimate(7, 2, 80, 4096, 32, 4096, 50), indent=2))

print("\nSame model QUANTIZED to int8 (half the weight memory):")
print(json.dumps(serving_estimate(7, 1, 80, 4096, 32, 4096, 60), indent=2))

7B model, fp16, on an 80GB GPU, 4096 ctx:
{
  "weights_GB": 14.0,
  "kv_per_seq_GB": 2.147,
  "free_for_kv_GB": 64.0,
  "max_concurrent_seqs": 29,
  "aggregate_tok_per_s": 1450
}

Same model QUANTIZED to int8 (half the weight memory):
{
  "weights_GB": 7.0,
  "kv_per_seq_GB": 1.074,
  "free_for_kv_GB": 71.0,
  "max_concurrent_seqs": 66,
  "aggregate_tok_per_s": 3960
}


Notice quantization doesn't just shrink weights — it frees memory for *more
concurrent KV-caches*, which raises aggregate throughput. Every lever in this
notebook (KV-cache, quantization, continuous batching, context length) feeds this
one equation, which is what actually determines your $/token in production.

## 8. 🏭 In practice

| You built | Real-world equivalent |
|-----------|----------------------|
| KV-cache loop | the cache in every transformer inference engine |
| `kv_cache_bytes` + paging discussion | vLLM PagedAttention |
| `quantize_int8` | GPTQ, AWQ, SmoothQuant, bitsandbytes, `llama.cpp` GGUF |
| continuous batching simulator | vLLM, HuggingFace TGI, TensorRT-LLM in-flight batching |
| `serving_estimate` | capacity planning every LLM infra team does |

In production you'd reach for **vLLM** or **TGI** rather than implement these — but
when vLLM's docs talk about *PagedAttention*, *continuous batching*, *KV-cache
utilisation*, or *quantized serving*, you now know exactly what each one is doing
and why, because you built the core of each by hand.

```python
# The whole of this notebook, in production, is roughly:
# from vllm import LLM, SamplingParams
# llm = LLM(model="meta-llama/Llama-2-7b-hf", quantization="awq",
#           gpu_memory_utilization=0.9, max_num_seqs=64)   # continuous batching + KV mgmt
# out = llm.generate(prompts, SamplingParams(max_tokens=256))
# (PagedAttention, KV-cache, and in-flight batching are all handled internally.)
```

---

## 🎯 Exercises

Try these before moving on. They are deliberately open-ended — the goal is to
extend the machinery you just built, not to find a single right answer.

1. Extend the KV-cache to *multi-head* attention (split D into H heads). Show the memory formula gains a factor for heads and confirm the O(N) generation cost is unchanged.
2. Implement int4 quantization with per-channel (per-column) scales instead of per-tensor. Measure the reconstruction-error improvement. Why do per-channel scales matter more at lower bit-widths?
3. Add a `max_num_seqs` cap and a waiting queue to the continuous-batching simulator, then plot throughput vs. TTFT as you raise the cap. Where's the knee? (This is the Week 3 latency/throughput frontier, for LLMs.)
4. Combine everything: estimate the $/1M-tokens for serving a 13B int8 model on a GPU costing $2/hr at the utilisation your simulator predicts. Which lever (quantization, batching, context length) moves cost the most?
5. PagedAttention avoids KV-cache fragmentation. Simulate fragmentation: allocate contiguous KV blocks for sequences of random length, free them randomly, and show how much memory is wasted vs. a paged allocator. Connect this to OS virtual memory from Week 2.

---

## ✅ Recap — Week 6

- Autoregressive generation makes LLM serving uniquely hard: N sequential passes, two latencies (TTFT/ITL), unknown output length.
- The KV-cache turns O(N^2) generation into O(N) and is the dominant memory consumer — context length is expensive.
- Quantization (int8/int4) halves or quarters weight memory at small quality cost, freeing room for more concurrent requests.
- Continuous (iteration-level) batching keeps the GPU full despite variable output lengths — the big throughput win over static batching.
- All these levers feed one capacity/cost equation; vLLM and TGI implement exactly what you built here.

### Coming up next

**Course complete.** You've climbed the MLOps maturity ladder from a notebook to a self-healing, monitored, LLM-capable deployment stack — building the core of every tool from first principles. See the capstone in the README to integrate all six weeks into one end-to-end system.